In [6]:
import pandas as pd
import numpy as np

In [7]:
df = pd.read_csv('../data/unified_features/unified_features_final.csv', low_memory = False)
kriwacki_split_df = pd.read_csv('../data/kriwacki_split_with_parent.csv', low_memory = False)
condenseq_df = pd.read_csv("../data/condenseq_data/complete_dataset_with_new_features.csv",low_memory = False)
split_df = pd.read_csv("../data/condenseq_data/medium_GFP_remove_top_half_high_discrepancy.csv", low_memory = False) 

In [8]:
kriwacki_split_df = kriwacki_split_df.rename(columns = {'idr_aaseq': 'protein_seq'})

In [9]:
# entire subset dataframe with kriwacki sequences
main_df = df.merge(split_df[['protein_seq', 'base_split']], on='protein_seq', how='left')
main_df = main_df.merge(condenseq_df[['protein_seq', 'protein_id']], on='protein_seq', how='left')
main_df = main_df.merge(kriwacki_split_df[['protein_seq', 'split', 'cluster_id']], on='protein_seq', how='left')
main_df['base_split'] = main_df['base_split'].fillna(main_df['split'])
main_df['protein_id'] = main_df['protein_id'].fillna(main_df['cluster_id'])
main_df = main_df.dropna(subset = ['base_split'])
main_df = main_df.dropna(subset = ['protein_id'])

In [10]:
# subset dataframe with just condenseq sequences
condenseq_subset_df = df.merge(split_df[['protein_seq', 'base_split']], on = 'protein_seq', how = 'left')
condenseq_subset_df = condenseq_subset_df.merge(condenseq_df[['protein_seq', 'protein_id']], on = 'protein_seq', how = 'left')
condenseq_subset_df = condenseq_subset_df.dropna(subset = ['base_split'])
condenseq_subset_df = condenseq_subset_df.dropna(subset = ['protein_id'])

In [11]:
main_df = main_df.drop(columns=['split', 'cluster_id'])

In [12]:
main_df.columns

Index(['protein_seq', 'fraction_A', 'fraction_C', 'fraction_D', 'fraction_E',
       'fraction_F', 'fraction_G', 'fraction_H', 'fraction_I', 'fraction_K',
       ...
       'frac_sticker', 'frac_spacer', 'sticker_spacer_ratio',
       'max_sticker_spacing', 'meta_disorder_mean', 'meta_disorder_max',
       'medium_GFP_fraction_cells_with_condensates', 'calvados_ah_pairs',
       'base_split', 'protein_id'],
      dtype='str', length=110)

In [14]:
main_df.to_csv('../data/unified_features/unified_features_final_subset.csv')
condenseq_subset_df.to_csv('../data/unified_features/unified_features_final_subset_og_seqs.csv')

In [11]:
# write fasta file for sequences
sequences = main_df['protein_seq'].tolist()

with open("../data/unified_features/unified_sequences.fasta", "w") as f:
    for index, seq in enumerate(sequences, start=1):
        # Write the header line followed by the sequence line
        f.write(f">seq_{index}\n")
        f.write(f"{seq}\n")